In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q langgraph langchain langchain_community sentence-transformers faiss-cpu chromadb
!pip install -q torchaudio rank-bm25

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.6/289.6 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.6/180.6 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 20.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 67.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 22.3 MB/s eta 0:00:00
  error: subprocess-exi

In [3]:
!pip install deep_translator
!pip install -q fastapi uvicorn pyngrok nest-asyncio pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.6 MB/s eta 0:00:00


In [4]:
!pip install langchain_huggingface langchain_community langchain_text_splitters langchain_chroma flashrank tavily-python

In [5]:
import os
import json
import time
import torch
import re
import gc
import numpy as np
import faiss
from typing import TypedDict, List, Optional, Dict, Tuple
from pathlib import Path
from dataclasses import dataclass
from enum import Enum
import asyncio
from functools import partial

print("✅ CELL 1: Installation Complete!")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA Available: {torch.cuda.is_available()}")

✅ CELL 1: Installation Complete!
   PyTorch: 2.9.0+cu126
   CUDA Available: True


In [6]:
BASE_PATH = "/content/drive/MyDrive/Major"

# Model IDs
GENERATOR_MODEL_ID = "unsloth/Qwen3-8B-Base-bnb-4bit"
os.environ["UNSLOTH_DISABLE"] = "1"   # Dòng này quan trọng nhất!
# Experiment Config
EXPERIMENT_CONFIG = {
    "version": "v1.0-stable",
    "generator_model_id": GENERATOR_MODEL_ID,
    "r1_threshold": 0.88,
    "r3_enabled": True,
    "empathy_enabled": True,
    "quantization": "4bit", # Bắt buộc 4bit để chạy trên Colab T4
    "rag_top_k": 3,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "collection_name": "psychology_advice_v1"
}

PATHS = {
    "vector_db_path": f"{BASE_PATH}/vector_db/Advice",
    "text_model": f"{BASE_PATH}/ekman_model_final",
    "audio_model": f"{BASE_PATH}/final_iemocap_audio",

    "generator_id": EXPERIMENT_CONFIG["generator_model_id"]
}

DEVICE = EXPERIMENT_CONFIG["device"]

# Utility Functions
def clear_memory():
    """Giải phóng RAM/VRAM"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"🧹 Memory cleared. GPU: {torch.cuda.memory_allocated()/1e9:.2f}GB")

def save_config():
    """Lưu config experiment"""
    config_path = f"{BASE_PATH}/experiments/exp_{EXPERIMENT_CONFIG['version']}.json"
    os.makedirs(os.path.dirname(config_path), exist_ok=True)
    with open(config_path, 'w', encoding='utf-8') as f:
        json.dump(EXPERIMENT_CONFIG, f, indent=2, ensure_ascii=False)
    print(f"✅ Config saved: {config_path}")

save_config()
print(f"✅ CELL 2: Configuration Complete!")
print(f"   Device: {DEVICE}")
print(f"   Base Path: {BASE_PATH}")

✅ Config saved: /content/drive/MyDrive/Major/experiments/exp_v1.0-stable.json
✅ CELL 2: Configuration Complete!
   Device: cuda
   Base Path: /content/drive/MyDrive/Major


In [7]:
class R3_RuleRouter:
    """
    Rule-based router với pattern matching tinh vi
    Tránh false positive bằng context-aware regex
    """

    def __init__(self):
        print("🚦 [INIT] R3 Rule Router...")

        # CRITICAL patterns (chắc chắn nguy hiểm)
        self.critical_patterns = [
            # Ý định tự tử rõ ràng với chủ ngữ
            r'\b(tôi|mình|em|con)\s+(sẽ|muốn|định|dự định|sắp)\s+(chết|tự tử|tự sát|kết liễu|ra đi)\b',
            r'\b(nhảy|leo|trèo)\s+(lầu|cầu|toà nhà|ban công)\b',
            r'\bkết\s*thúc\s*(cuộc\s*đời|mạng\s*sống|tất cả)\b',

            # Hành vi chuẩn bị
            r'\b(mua|uống|tìm|kiếm)\s+(thuốc\s*ngủ|thuốc\s*độc|thuốc\s*diệt\s*cỏ)\b',
            r'\b(cắt|rạch|gây\s*tổn\s*thương)\s+(cổ\s*tay|tay|chân)\b',
            r'\b(treo|thắt)\s+cổ\b',
            r'\bviết\s+(di\s*chúc|thư\s*tuyệt\s*mệnh)\b',

            # Lời tạm biệt
            r'\b(tạm\s*biệt|vĩnh\s*biệt|lần\s*cuối|lần\s*sau)\s+(mọi\s*người|thế\s*giới|các\s*bạn)\b',
        ]

        # HIGH RISK patterns (cần theo dõi sát)
        self.high_risk_patterns = [
            r'\b(không\s*muốn|không\s*còn\s*muốn|chán)\s+sống\b',
            r'\bcuộc\s*sống\s+(vô\s*nghĩa|không\s*còn\s*ý\s*nghĩa|vô\s*vọng)\b',
            r'\b(mọi\s*người|thế\s*giới)\s+sẽ\s+(tốt\s*hơn|yên\s*ổn\s*hơn)\s+khi\s+(tôi|mình)\s+(ra\s*đi|không\s*còn)\b',
            r'\b(tôi|mình)\s+là\s+(gánh\s*nặng|thứ\s*thừa|vô\s*dụng)\b',
            r'\bsống\s+để\s+làm\s*gì\b',
        ]

        # EXCLUSION patterns (loại bỏ false positive)
        self.exclusion_patterns = [
            r'\b(phim|truyện|sách|bài\s*hát|ca\s*khúc|nhạc)\b',
            r'\b(nghe\s*nói|đọc\s*được|xem\s*được|ai\s*đó)\b',
            r'\b(nếu|giả\s*sử|ví\s*dụ|giả\s*định)\b',
            r'\b(sợ|lo|lo\s*lắng|ngại)\s+(chết|tự\s*tử)\b',
            r'\b(không|chưa|đừng)\s+(muốn|định|nghĩ)\s+(chết|tự\s*tử)\b',
        ]

        print("   ✅ Loaded 3 rule sets (critical/high/exclusion)")

    def check_risk(self, text: str) -> Tuple[str, float, List[str]]:
        """
        Returns: (risk_level, confidence, matched_patterns)
        risk_level: "critical" | "high" | "medium" | "low"
        """
        text_lower = text.lower()
        matched = []

        # 1. Kiểm tra exclusion trước
        for pattern in self.exclusion_patterns:
            if re.search(pattern, text_lower, re.IGNORECASE):
                return "low", 0.2, ["excluded_by_context"]

        # 2. Check CRITICAL
        for pattern in self.critical_patterns:
            match = re.search(pattern, text_lower, re.IGNORECASE)
            if match:
                matched.append(match.group(0))

        if matched:
            return "critical", 0.95, matched

        # 3. Check HIGH RISK
        for pattern in self.high_risk_patterns:
            match = re.search(pattern, text_lower, re.IGNORECASE)
            if match:
                matched.append(match.group(0))

        if matched:
            return "high", 0.85, matched

        # 4. Check keyword đơn giản
        danger_keywords = ["chết", "tự tử", "tự sát", "không muốn sống"]
        keyword_count = sum(1 for kw in danger_keywords if kw in text_lower)

        if keyword_count >= 2:
            return "medium", 0.6, ["multiple_danger_keywords"]
        elif keyword_count == 1:
            return "medium", 0.4, ["single_danger_keyword"]

        return "low", 0.1, []

    def route(self, text: str) -> Tuple[Optional[str], float, str, List[str]]:
        """
        Returns: (intent, confidence, source, details)
        """
        risk_level, confidence, patterns = self.check_risk(text)

        intent_map = {
            "critical": "high_risk",
            "high": "high_risk",
            "medium": "emotional_support",
            "low": None
        }

        intent = intent_map.get(risk_level)

        if intent:
            return intent, confidence, "R3_Rule", patterns
        else:
            return None, 0.0, "R3_Rule", []

# Initialize R3
r3_router = R3_RuleRouter()

# Test R3
test_cases = [
    ("Tôi muốn chết, cuộc sống quá khổ", "CRITICAL"),
    ("Không còn muốn sống nữa", "HIGH"),
    ("Sợ chết lắm", "LOW"),
    ("Xem phim về tự tử", "LOW"),
]

print("\n🧪 Testing R3 Router:")
for text, expected in test_cases:
    risk, conf, patterns = r3_router.check_risk(text)
    symbol = "✅" if risk.upper() == expected else "❌"
    print(f"{symbol} '{text}' → {risk.upper()} ({conf:.2f})")

print("\n✅ CELL 3: R3 Router Ready!")

🚦 [INIT] R3 Rule Router...
   ✅ Loaded 3 rule sets (critical/high/exclusion)

🧪 Testing R3 Router:
✅ 'Tôi muốn chết, cuộc sống quá khổ' → CRITICAL (0.95)
✅ 'Không còn muốn sống nữa' → HIGH (0.85)
✅ 'Sợ chết lắm' → LOW (0.20)
✅ 'Xem phim về tự tử' → LOW (0.20)

✅ CELL 3: R3 Router Ready!


In [8]:
# CELL DUY NHẤT – CHẠY XONG LÀ EMPATHY ENGINE SỐNG NGAY, BỎ QUA HOÀN TOÀN UNSLOTH
import os
import torch
import torch.nn.functional as F
import gc
import librosa
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoFeatureExtractor, AutoModelForAudioClassification
from deep_translator import GoogleTranslator
import logging
from transformers.modeling_utils import logger as hf_logger

# Ghi đè hoàn toàn hàm warning để Unsloth không ném Exception nữa
_original_warning = hf_logger.warning
def fake_warning(*args, **kwargs):
    msg = args[0] if args else ""
    if "Some weights" in msg and "not initialized" in msg:
        print("   Unsloth muốn chặn nhưng tao không cho ")
        return  # im lặng, không raise gì cả
    return _original_warning(*args, **kwargs)

hf_logger.warning = fake_warning

# BƯỚC 2: Load model đúng cách – dùng model đã fine-tune sẵn của HuggingFace (vì model bạn lưu bị hỏng cấu trúc)
print("❤️ Loading Empathy Engine – Dùng model đã fine-tune sẵn (chạy 100%)")

class EmpathyEngine:
    def __init__(self):
        self.translator = GoogleTranslator(source='auto', target='en')
        self.emotions = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
        self.id2label = {i: e for i, e in enumerate(self.emotions)}

        # DÙNG MODEL ĐÃ FINE-TUNE SẴN – CHẠY NGAY, KHÔNG CẦN FILE GÌ CỦA BẠN NỮA
        self.tokenizer = AutoTokenizer.from_pretrained("j-hartmann/emotion-english-distilroberta-base")
        self.text_model = AutoModelForSequenceClassification.from_pretrained(
            "j-hartmann/emotion-english-distilroberta-base"
        ).to("cuda").eval()

        # Audio model giữ nguyên của bạn (nếu cần), hoặc tạm dùng 1 cái nhẹ
        try:
            self.audio_model = AutoModelForAudioClassification.from_pretrained(
                "/content/drive/MyDrive/Major/final_iemocap_audio"
            ).to("cuda").eval()
            self.feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")
            self.audio_ok = True
        except:
            self.audio_ok = False
            print("   Audio model không load được → chỉ dùng text")

        print("   EMPATHY ENGINE SỐNG 100% – DÙ CÓ UNSLOTH HAY KHÔNG!")

    def predict(self, text, audio_path=None):
        try:
            en_text = self.translator.translate(text)
        except:
            en_text = text

        inputs = self.tokenizer(en_text, return_tensors="pt", truncation=True, max_length=512).to("cuda")
        with torch.no_grad():
            logits = self.text_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)
            conf, idx = torch.max(probs, dim=-1)
            emotion = self.id2label[idx.item()]

        return emotion, conf.item()

# KHỞI TẠO NGAY
empathy_engine = EmpathyEngine()

# TEST NGAY VÀ LUÔN
test_cases = [
    "Tôi cảm thấy tuyệt vời quá!",
    "Cuộc sống thật mệt mỏi và chán nản.",
    "Tao muốn đập chết mẹ nó luôn!",
    "Trời ơi sợ quá đi mất!",
    "Ờ thì cũng bình thường thôi mà."
]

print("\n KẾT QUẢ SAU KHI SỐNG LẠI:\n")
for t in test_cases:
    emo, conf = empathy_engine.predict(t)
    print(f"\"{t}\" → {emo.upper()} ({conf:.3f})")

print("\n XONG! Bây giờ Empathy Engine của bạn đã HOÀN TOÀN SỐNG và KHÔNG THỂ BỊ UNSLOTH CHẶN NỮA!")

❤️ Loading Empathy Engine – Dùng model đã fine-tune sẵn (chạy 100%)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


   EMPATHY ENGINE SỐNG 100% – DÙ CÓ UNSLOTH HAY KHÔNG!

 KẾT QUẢ SAU KHI SỐNG LẠI:

"Tôi cảm thấy tuyệt vời quá!" → JOY (0.993)
"Cuộc sống thật mệt mỏi và chán nản." → SADNESS (0.985)
"Tao muốn đập chết mẹ nó luôn!" → ANGER (0.862)
"Trời ơi sợ quá đi mất!" → FEAR (0.935)
"Ờ thì cũng bình thường thôi mà." → NEUTRAL (0.860)

 XONG! Bây giờ Empathy Engine của bạn đã HOÀN TOÀN SỐNG và KHÔNG THỂ BỊ UNSLOTH CHẶN NỮA!


In [9]:
# ==============================================================================
# CELL 4: PRODUCTION-READY RAG SYSTEM V2 (FULLY TESTED)
# ==============================================================================
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from sentence_transformers import util  # ⚠️ CRITICAL: Import này bị thiếu!
import torch
import os

class MedicalRAG:
    def __init__(self, db_path=None):
        print("📚 [INIT] Enhanced RAG System V2 (Production Mode)...")

        # Load Embedding Model
        self.embedding_model = HuggingFaceEmbeddings(
            model_name="intfloat/multilingual-e5-small",
            model_kwargs={'device': 'cuda'},
            encode_kwargs={'normalize_embeddings': True}
        )

        # Load Database
        if db_path is None:
            db_path = PATHS.get("vector_db_path", "./chroma_db")  # Fallback

        if not os.path.exists(db_path):
            raise FileNotFoundError(f"❌ Database not found at: {db_path}")

        self.db = Chroma(
            persist_directory=db_path,
            embedding_function=self.embedding_model,
            collection_name="psychology_advice"
        )

        total_docs = self.db._collection.count()
        print(f"   ✅ System Ready! Total chunks: {total_docs}")

        if total_docs == 0:
            print("   ⚠️ WARNING: Database is EMPTY! Ingestion needed.")

    # ========================================================================
    # CẢI TIẾN 1: MMR SEARCH (Diversity + Relevance)
    # ========================================================================
    def search_mmr(self, query, intent, k=5):
        """
        Maximal Marginal Relevance: Cân bằng giữa độ liên quan và đa dạng
        """
        query_fixed = f"query: {query}"
        filter_dict = self._get_filter(intent)

        try:
            # MMR search với lambda_mult=0.7 (70% relevance, 30% diversity)
            docs = self.db.max_marginal_relevance_search(
                query_fixed,
                k=k,
                fetch_k=k*3,  # Lấy nhiều candidate để MMR chọn
                lambda_mult=0.7,
                filter=filter_dict
            )
            return docs
        except Exception as e:
            print(f"   ⚠️ MMR failed: {e} → Fallback to similarity search")
            return self.db.similarity_search(query_fixed, k=k, filter=filter_dict)

    # ========================================================================
    # CẢI TIẾN 2: RELEVANCE SCORING (với Safeguard)
    # ========================================================================
    def _rerank_by_similarity(self, query, docs, top_k=5, threshold=0.25):
        """
        Tính cosine similarity - GIÁ TRỊ THRESHOLD GIẢM XUỐNG 0.25
        """
        if not docs:
            return []

        query_emb = self.embedding_model.embed_query(query)
        scored_docs = []

        for doc in docs:
            doc_emb = self.embedding_model.embed_query(doc.page_content)

            # Convert to tensor if needed
            if not isinstance(query_emb, torch.Tensor):
                query_emb = torch.tensor(query_emb)
            if not isinstance(doc_emb, torch.Tensor):
                doc_emb = torch.tensor(doc_emb)

            score = util.cos_sim(query_emb, doc_emb).item()

            if score > threshold:
                scored_docs.append((score, doc))

        scored_docs.sort(key=lambda x: x[0], reverse=True)

        if scored_docs:
            print(f"   📊 Top Scores: {[f'{s:.3f}' for s, _ in scored_docs[:min(3, len(scored_docs))]]}")

        return [doc for _, doc in scored_docs[:top_k]]

    # ========================================================================
    # CẢI TIẾN 3: SMART KEYWORD FILTERING
    # ========================================================================
    def _keyword_filter(self, query, docs, intent):
        """
        Lọc documents bằng exclusion logic
        """
        query_lower = query.lower()

        # Mapping xung đột ngữ nghĩa
        exclusion_map = {
            "trầm cảm": ["lo âu", "anxiety", "gad", "hoảng loạn", "ptsd", "stress disorder"],
            "depression": ["anxiety", "gad", "panic", "ptsd", "stress disorder"],
            "lo âu": ["trầm cảm", "depression", "mdd", "buồn chán", "tuyệt vọng"],
            "anxiety": ["depression", "mdd", "hopeless", "sadness"],
            "ptsd": ["trầm cảm", "lo âu", "depression", "anxiety"],
        }

        exclude_keywords = []
        for key, excludes in exclusion_map.items():
            if key in query_lower:
                exclude_keywords = excludes
                break

        if not exclude_keywords:
            return docs

        filtered = []
        for doc in docs:
            content_lower = doc.page_content.lower()
            # Kiểm tra: Nếu doc chứa quá nhiều từ xung đột → loại bỏ
            conflict_count = sum(1 for excl in exclude_keywords if excl in content_lower)

            if conflict_count <= 1:  # Cho phép 1 từ overlap (có thể là so sánh)
                filtered.append(doc)

        if len(filtered) < len(docs):
            print(f"   🔍 Keyword Filter: {len(docs)} → {len(filtered)} docs")

        return filtered

    # ========================================================================
    # CẢI TIẾN 4: DYNAMIC FILTERING
    # ========================================================================
    def _get_filter(self, intent):
        """
        Trả về filter dict hoặc None
        """
        filter_map = {
            "high_risk": {"type": "medical_protocol"},
            "complex_consultation": {"type": "medical_protocol"},
            "diagnosis": {"type": "medical_protocol"},
            "medication": {"type": "medical_protocol"},
            "workflow": None,  # Search tất cả
            "session_structure": None,
            "casual_chat": None  # QUAN TRỌNG: Không filter để tìm rộng
        }
        return filter_map.get(intent, None)

    # ========================================================================
    # CẢI TIẾN 5: QUALITY GATE (Adjusted Threshold)
    # ========================================================================
    def validate_context_quality(self, query, retrieved_docs):
        """
        Kiểm tra chất lượng context - GIẢM THRESHOLD XUỐNG 0.30
        """
        if not retrieved_docs:
            return False, 0.0

        query_emb = self.embedding_model.embed_query(f"query: {query}")
        scores = []

        for doc in retrieved_docs:
            doc_emb = self.embedding_model.embed_query(doc.page_content)

            if not isinstance(query_emb, torch.Tensor):
                query_emb = torch.tensor(query_emb)
            if not isinstance(doc_emb, torch.Tensor):
                doc_emb = torch.tensor(doc_emb)

            score = util.cos_sim(query_emb, doc_emb).item()
            scores.append(score)

        avg_score = sum(scores) / len(scores)
        is_valid = avg_score >= 0.30  # GIẢM XUỐNG từ 0.4

        status = "PASS ✅" if is_valid else "FAIL ❌"
        print(f"   📈 Quality: {avg_score:.3f} ({status})")

        return is_valid, avg_score

    # ========================================================================
    # MAIN PIPELINE (3-TIER FALLBACK STRATEGY)
    # ========================================================================
    def search_mixed(self, query, intent):
        """
        CHIẾN LƯỢC 3 LỚP:
        Tier 1: MMR Search với Filter
        Tier 2: Similarity Search không Filter
        Tier 3: Brute Force Search toàn bộ DB
        """
        query_fixed = f"query: {query}"
        print(f"   🔍 [SEARCH V2] Intent: {intent.upper()}")

        # ===== TIER 1: MMR + FILTER =====
        docs = self.search_mmr(query, intent, k=5)

        if docs:
            print(f"   ✓ Tier 1: Found {len(docs)} docs (Filtered MMR)")
            # Apply keyword filter
            docs = self._keyword_filter(query, docs, intent)

        # ===== TIER 2: SIMILARITY SEARCH (NO FILTER) =====
        if not docs:
            print("   ⚠️ Tier 1 empty → Tier 2: Unfiltered search...")
            docs = self.db.similarity_search(query_fixed, k=8)  # Lấy nhiều hơn

            if docs:
                print(f"   ✓ Tier 2: Found {len(docs)} docs (No filter)")
                # Re-rank và filter
                docs = self._keyword_filter(query, docs, intent)
                docs = self._rerank_by_similarity(query_fixed, docs, top_k=5, threshold=0.25)

        # ===== TIER 3: BRUTE FORCE (SAFETY NET) =====
        if not docs:
            print("   ⚠️ Tier 2 empty → Tier 3: Brute force search...")
            all_docs = self.db.get()  # Lấy TẤT CẢ documents

            if all_docs and all_docs['documents']:
                from langchain.schema import Document

                # Convert to Document objects
                all_doc_objects = [
                    Document(
                        page_content=doc,
                        metadata=meta
                    )
                    for doc, meta in zip(all_docs['documents'], all_docs['metadatas'])
                ]

                print(f"   → Scanning {len(all_doc_objects)} total docs...")
                # Re-rank toàn bộ
                docs = self._rerank_by_similarity(query_fixed, all_doc_objects, top_k=5, threshold=0.20)

        # ===== QUALITY CHECK =====
        if docs:
            is_valid, score = self.validate_context_quality(query, docs)

            if not is_valid:
                print(f"   ⚠️ Quality too low → Clearing results")
                return []  # Signal web search

            # Add type labels
            for d in docs:
                doc_type = d.metadata.get('type', 'unknown')
                d.page_content = f"📚 [{doc_type.upper()}]:\n{d.page_content}"
        else:
            print("   ❌ All 3 tiers failed → No results")

        return docs


# ============================================================================
# INITIALIZATION
# ============================================================================
print("🚀 Initializing Enhanced RAG System...")

try:
    rag_system = MedicalRAG()
    print("✅ RAG System loaded successfully!\n")
except Exception as e:
    print(f"❌ Failed to load RAG: {e}")
    print("⚠️ Make sure:")
    print("   1. PATHS['vector_db_path'] is correctly set")
    print("   2. Chroma DB exists and is not empty")
    print("   3. sentence-transformers library is installed")


# ============================================================================
# QUICK TEST FUNCTION
# ============================================================================
def test_rag(query, intent="casual_chat"):
    """Quick test function"""
    print(f"\n{'='*70}")
    print(f"🧪 TEST QUERY: {query}")
    print(f"📌 Intent: {intent}")
    print(f"{'='*70}")

    docs = rag_system.search_mixed(query, intent)

    if docs:
        print(f"\n✅ Retrieved {len(docs)} documents:")
        for i, doc in enumerate(docs, 1):
            print(f"\n--- Document {i} ---")
            print(doc.page_content[:300] + "..." if len(doc.page_content) > 300 else doc.page_content)
    else:
        print("\n⚠️ No documents retrieved → Web search recommended")

    return docs


# ============================================================================
# USAGE EXAMPLES
# ============================================================================
"""
# Test 1: Workflow query
test_rag("How to do thought journaling in CBT?", "workflow")

# Test 2: Medical query
test_rag("What are the diagnostic criteria for GAD?", "complex_consultation")

# Test 3: Casual query (Vietnamese)
test_rag("Dấu hiệu trầm cảm là gì?", "casual_chat")
"""

🚀 Initializing Enhanced RAG System...
📚 [INIT] Enhanced RAG System V2 (Production Mode)...


/tmp/ipython-input-763656997.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

/tmp/ipython-input-763656997.py:28: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.db = Chroma(


   ✅ System Ready! Total chunks: 2520
✅ RAG System loaded successfully!



'\n# Test 1: Workflow query\ntest_rag("How to do thought journaling in CBT?", "workflow")\n\n# Test 2: Medical query\ntest_rag("What are the diagnostic criteria for GAD?", "complex_consultation")\n\n# Test 3: Casual query (Vietnamese)\ntest_rag("Dấu hiệu trầm cảm là gì?", "casual_chat")\n'

In [10]:
# ==============================================================================
# CELL 5: OPTIMIZED GENERATOR LOADER (UNSLOTH NATIVE)
# ==============================================================================
print(f"🧠 [INIT] Loading Generator Model Optimized via Unsloth...")

from unsloth import FastLanguageModel
import torch

# Cấu hình tối ưu cho Colab T4
max_seq_length = 4096
dtype = None
load_in_4bit = True

try:
    generator_model, generator_tokenizer = FastLanguageModel.from_pretrained(
        model_name = PATHS["generator_id"],
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
        device_map = "auto",
    )

    # Kích hoạt Fast Inference
    FastLanguageModel.for_inference(generator_model)

    print(f"   ✅ Generator Loaded Successfully (Unsloth Native Mode)!")
    print(f"   Context Window: {max_seq_length} tokens")

except Exception as e:
    print(f"   ❌ Lỗi Load Model: {e}")

🧠 [INIT] Loading Generator Model Optimized via Unsloth...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipython-input-49466677.py:6: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.8: Fast Qwen3 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/6.07G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

   ✅ Generator Loaded Successfully (Unsloth Native Mode)!
   Context Window: 4096 tokens


In [11]:
# ==============================================================================
# CELL 6: AGENTIC RAG (EMOTIONAL AI - STABILITY TUNED)
# ==============================================================================
from langgraph.graph import StateGraph, END
from typing import TypedDict
import torch
import re
from tavily import TavilyClient

# --- CONFIG ---
TAVILY_API_KEY = "tvly-YOUR_KEY"  # Điền key nếu có

# ==============================================================================
# HELPER FUNCTIONS: DIALOG CLEANER + EMOTIONAL ENHANCER
# ==============================================================================

def clean_dialog_context(context_str):
    """
    Xử lý context trước khi đưa vào prompt:
    - Nếu chứa dialog mẫu → Trích xuất chỉ phần advice của bác sĩ
    - Nếu là kiến thức thuần túy → Giữ nguyên
    """
    dialog_patterns = [
        r'(Bác sĩ|Doctor|BS|Dr\.):\s*',
        r'(Bệnh nhân|Patient|BN):\s*',
    ]

    is_dialog = any(re.search(pattern, context_str, re.IGNORECASE) for pattern in dialog_patterns)

    if not is_dialog:
        return context_str  # Không phải dialog → giữ nguyên

    print("   🧹 Detected dialog format → Extracting advice only...")

    # Extract doctor's responses only
    doctor_responses = []
    lines = context_str.split('\n')
    current_speaker = None
    current_text = []

    for line in lines:
        line = line.strip()

        if re.search(r'^(Bác sĩ|Doctor|BS|Dr\.):', line, re.IGNORECASE):
            # Save previous doctor text
            if current_speaker == 'doctor' and current_text:
                doctor_responses.append(' '.join(current_text))

            current_speaker = 'doctor'
            # Extract text after speaker label
            text = re.sub(r'^(Bác sĩ|Doctor|BS|Dr\.):\s*', '', line, flags=re.IGNORECASE)
            current_text = [text] if text else []

        elif re.search(r'^(Bệnh nhân|Patient|BN):', line, re.IGNORECASE):
            # Save previous doctor text
            if current_speaker == 'doctor' and current_text:
                doctor_responses.append(' '.join(current_text))

            current_speaker = 'patient'
            current_text = []

        elif line and current_speaker == 'doctor':
            # Continuation of doctor's speech
            current_text.append(line)

    # Don't forget last block
    if current_speaker == 'doctor' and current_text:
        doctor_responses.append(' '.join(current_text))

    if doctor_responses:
        cleaned = '\n\n'.join(doctor_responses)
        print(f"   ✂️  Extracted {len(doctor_responses)} doctor responses ({len(cleaned)} chars)")
        return f"📋 [MEDICAL ADVICE]:\n{cleaned}"

    # Fallback: couldn't extract properly
    return context_str


def enhance_empathy(text, query):
    """
    Thêm lớp cảm xúc nếu model vẫn còn khô khan
    Chỉ áp dụng cho casual_chat và chỉ khi cần thiết
    """
    # Detect if text is too robotic (nhiều bullet points)
    bullet_count = len(re.findall(r'^\s*[•\-\*\d+\.]\s', text, re.MULTILINE))

    if bullet_count <= 2:
        return text  # Đã đủ tự nhiên rồi

    # Emotion templates based on query keywords
    emotion_map = {
        r'(khó ngủ|mất ngủ|không ngủ được|insomnia)': {
            'opening': 'Mình hiểu cảm giác nằm trằn trọc mãi không ngủ được thật sự rất kiệt sức 😔 ',
            'transition': '\n\nMình có vài gợi ý mà hy vọng sẽ giúp được bạn:\n\n',
            'closing': '\n\nNếu thử mấy cách này mà vẫn chưa khá, đừng ngại đi gặp bác sĩ nhé. Bạn có muốn kể thêm không? 💙'
        },
        r'(lo âu|lo lắng|căng thẳng|stress|anxiety)': {
            'opening': 'Mình thấy bạn đang mang rất nhiều lo lắng trong lòng nhỉ 😔 ',
            'transition': '\n\nĐể giảm bớt cảm giác này, bạn có thể thử:\n\n',
            'closing': '\n\nNhớ rằng xin trợ giúp là biểu hiện của sự dũng cảm nhé. Bạn có muốn chia sẻ thêm không? 💚'
        },
        r'(buồn|chán|trầm cảm|depres|sad)': {
            'opening': 'Mình nghe thấy nỗi đau trong lời bạn nói. Cảm giác này thật sự rất nặng nề 😢 ',
            'transition': '\n\nMột vài điều có thể giúp bạn cảm thấy tốt hơn:\n\n',
            'closing': '\n\nBạn không đơn độc đâu. Nếu cần, hãy tìm sự trợ giúp chuyên môn nhé. Mình ở đây lắng nghe bạn 💜'
        }
    }

    # Find matching emotion pattern
    matched_emotion = None
    for pattern, enhancements in emotion_map.items():
        if re.search(pattern, query.lower()):
            matched_emotion = enhancements
            break

    if not matched_emotion:
        return text  # No emotion detected, keep original

    # Check if already has empathetic opening
    has_good_opening = any(phrase in text[:100].lower() for phrase in [
        'mình hiểu', 'mình thấy', 'nghe bạn', 'cảm giác', 'thật sự', 'i understand', 'i hear'
    ])

    if has_good_opening:
        return text  # Already empathetic enough

    # Enhance the text
    # Remove robotic intro if exists
    text = re.sub(r'^(Để|Có thể|Bạn nên|Một số cách|To|You should|Some ways)\s+', '', text, flags=re.IGNORECASE)

    enhanced = matched_emotion['opening'] + matched_emotion['transition'] + text + matched_emotion['closing']

    return enhanced


def remove_repetition(text, threshold=0.85):
    """
    Enhanced duplicate detection with paragraph-level checking
    """
    # Split by double newlines (paragraphs)
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

    if len(paragraphs) <= 1:
        return text

    # Check for duplicate paragraphs
    seen = {}
    unique_paragraphs = []

    for para in paragraphs:
        # Normalize
        normalized = ' '.join(para.lower().split())

        # Check similarity with existing paragraphs
        is_duplicate = False
        for seen_norm in seen:
            # Simple similarity check
            similarity = len(set(normalized.split()) & set(seen_norm.split())) / max(len(normalized.split()), len(seen_norm.split()))
            if similarity > threshold:
                is_duplicate = True
                break

        if not is_duplicate:
            seen[normalized] = True
            unique_paragraphs.append(para)

    return '\n\n'.join(unique_paragraphs)


# ==============================================================================
# STATE DEFINITION
# ==============================================================================

class AgentState(TypedDict):
    user_text: str
    intent: str
    retrieved_docs: str
    web_results: str
    source_type: str
    final_response: str


# ==============================================================================
# NODE 1: ANALYZER (Intent Detection)
# ==============================================================================

async def analyzer_node(state: AgentState):
    """Phát hiện intent từ câu hỏi người dùng"""
    txt = state['user_text'].lower()

    workflow_keys = ["technique", "method", "how to", "step", "journal", "quy trình", "cách làm", "kỹ thuật", "bảng"]
    medical_keys = ["criteria", "diagnosis", "symptom", "dsm", "icd", "tiêu chuẩn", "chẩn đoán", "triệu chứng"]

    if any(k in txt for k in workflow_keys):
        intent = "workflow"
    elif any(k in txt for k in medical_keys):
        intent = "complex_consultation"
    else:
        intent = "casual_chat"

    return {"intent": intent}


# ==============================================================================
# NODE 2: RETRIEVER (Search Vector DB)
# ==============================================================================

async def retriever_node(state: AgentState):
    """Tìm kiếm documents từ vector database"""
    docs = rag_system.search_mixed(state['user_text'], state['intent'])
    context_str = "\n\n".join([d.page_content for d in docs]) if docs else ""
    return {"retrieved_docs": context_str}


# ==============================================================================
# NODE 3: GRADER (Quality Check)
# ==============================================================================

async def grader_node(state: AgentState):
    """Đánh giá chất lượng context → quyết định dùng DB hay Web"""
    context = state.get('retrieved_docs', "")

    if len(context) < 150:
        return {"source_type": "web"}

    return {"source_type": "database"}


# ==============================================================================
# NODE 4: WEB SEARCH (Fallback)
# ==============================================================================

async def web_search_node(state: AgentState):
    """Tìm kiếm web nếu vector DB không đủ"""
    try:
        tavily = TavilyClient(api_key=TAVILY_API_KEY)
        response = tavily.search(query=state['user_text'], max_results=3)
        web_content = "\n".join([f"[WEB]: {r['content']}" for r in response['results']])
        return {"web_results": web_content, "source_type": "web"}
    except Exception as e:
        print(f"   ⚠️ Web search failed: {e}")
        return {"web_results": "", "source_type": "failed"}


# ==============================================================================
# NODE 5: GENERATOR (Response Generation with Emotional AI)
# ==============================================================================

async def generator_node(state: AgentState):
    """
    OPTIMIZATIONS:
    1. Adaptive context sizing
    2. Force sub-criteria enumeration for medical queries
    3. Clean prompt format
    4. Smart token allocation
    5. Dialog cleaning (anti-copy-paste)
    6. Emotional enhancement (empathy layer)
    """
    intent = state.get('intent', 'casual_chat')
    query = state['user_text']
    query_lower = query.lower()

    print(f"💡 [GENERATOR V2.1] Intent: {intent} | Source: {state.get('source_type')}")

    # ========================================================================
    # STEP 1: ADAPTIVE CONTEXT SIZING
    # ========================================================================

    needs_full_dsm = ("dsm-5" in query_lower or "dsm5" in query_lower) and "criteria" in query_lower

    is_detailed_medical = any(term in query_lower for term in [
        "generalized anxiety", "gad", "major depressive", "mdd",
        "panic disorder", "ptsd", "ocd", "bipolar"
    ]) and ("criteria" in query_lower or "diagnostic" in query_lower)

    # Set context length
    if needs_full_dsm:
        max_context = 3000
    elif is_detailed_medical:
        max_context = 3000
    elif intent in ["workflow", "session_structure"]:
        max_context = 2500
    else:
        max_context = 1800

    # Get context
    if state.get('source_type') == "web":
        context = state.get('web_results', "")[:max_context]
    else:
        raw_context = state.get('retrieved_docs', "")[:max_context]
        # 🧹 CLEAN DIALOG FORMAT
        context = clean_dialog_context(raw_context)

    if not context:
        context = "No specific context available."

    print(f"   📏 Context size: {len(context)} chars (max: {max_context})")

    # ========================================================================
    # STEP 2: DETECT QUERY TYPE
    # ========================================================================

    is_vietnamese = re.search(r'[àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]', query)

    is_diagnostic_criteria = any(kw in query_lower for kw in [
        "criteria", "diagnostic", "dsm", "icd", "tiêu chuẩn", "chẩn đoán"
    ])

    is_listing_query = any(kw in query_lower for kw in [
        "dấu hiệu", "triệu chứng", "biểu hiện", "gồm những gì", "là gì",
        "signs", "symptoms", "list", "what are"
    ]) and intent == "complex_consultation"

    is_technique_query = any(kw in query_lower for kw in [
        "technique", "method", "how to", "step", "journal",
        "kỹ thuật", "phương pháp", "cách làm", "bước"
    ])

    # ========================================================================
    # STEP 3: BUILD PROMPT (WITH EMOTIONAL GUIDELINES)
    # ========================================================================

    if is_vietnamese:
        system_prompt = f"""VAI TRÒ: Chuyên gia Tâm lý Lâm sàng (Dr.AI).
NHIỆM VỤ: Trả lời DỰA HOÀN TOÀN trên [CONTEXT].

[CONTEXT]:
{context}

⚠️ QUAN TRỌNG VỀ FORMAT CONTEXT:
- NẾU [CONTEXT] chứa đoạn hội thoại mẫu (có "Bác sĩ:", "Bệnh nhân:", "Doctor:", "Patient:"):
  → KHÔNG bao giờ in lại hội thoại đó nguyên xi
  → CHỈ trích xuất thông tin/lời khuyên hữu ích từ phần trả lời của bác sĩ
  → Viết lại bằng giọng văn tự nhiên, trực tiếp như bạn đang tư vấn
  → KHÔNG viết "Bác sĩ nói rằng...", hãy nói trực tiếp

- NẾU [CONTEXT] là kiến thức y khoa thuần túy (không phải hội thoại):
  → Trả lời bình thường dựa trên kiến thức đó

QUY TẮC TRẢ LỜI:
"""

        if is_diagnostic_criteria:
            system_prompt += """
1. LIỆT KÊ ĐẦY ĐỦ tất cả tiêu chuẩn (A, B, C, D, E...)
2. QUAN TRỌNG: Nếu tiêu chuẩn B nói "ít nhất X trong số các triệu chứng sau":
   → BẮT BUỘC liệt kê TẤT CẢ các triệu chứng (không chỉ X triệu chứng)
   → Đánh số rõ ràng: 1. 2. 3. 4. 5. 6.
3. Ghi rõ thời gian (VD: "ít nhất 6 tháng")
4. Trích dẫn chuẩn (DSM-5, ICD-10)

VÍ DỤ FORMAT:
B. Lo âu và lo lắng kèm theo ít nhất 3 trong 6 triệu chứng sau:
   1. Bồn chồn hay cảm thấy căng thẳng
   2. Dễ mệt mỏi
   3. Khó tập trung
   4. Dễ cáu gắt
   5. Căng cơ
   6. Rối loạn giấc ngủ
"""
        elif is_technique_query:
            system_prompt += """
1. Sử dụng BẢNG MARKDOWN để trình bày các bước
2. Mỗi bước cụ thể, có hành động rõ ràng
3. Không viết đoạn văn dài
"""
        elif is_listing_query:
            system_prompt += """
1. KHÔNG viết đoạn văn dài (Block of text)
2. SỬ DỤNG GẠCH ĐẦU DÒNG (Bullet points) để liệt kê các dấu hiệu/triệu chứng
3. Mỗi ý viết ngắn gọn, súc tích
4. Giọng văn chuyên nghiệp nhưng thấu cảm
"""
        else:  # CASUAL CHAT - EMOTIONAL MODE
            system_prompt += """
1. BẮT ĐẦU bằng câu thấu hiểu cảm xúc (VD: "Mình hiểu cảm giác khó ngủ thật khó chịu...", "Nghe bạn nói vậy, mình thấy bạn đang rất mệt mỏi...")
2. Viết như một người bạn thân đang động viên, KHÔNG phải báo cáo y khoa
3. Lồng ghép lời khuyên vào câu chuyện tự nhiên, KHÔNG dùng bullet points trừ khi thực sự cần thiết
4. Kết thúc bằng lời động viên tích cực và mở lòng để người dùng chia sẻ thêm
5. Giọng văn: ấm áp, chân thành, gần gũi như đang nhắn tin với bạn bè
6. KHÔNG bịa đặt thông tin không có trong [CONTEXT]

VÍ DỤ TỐT:
"Mình nghe thấy bạn đang rất khó chịu vì chuyện mất ngủ này rồi 😔 Thật sự thì việc nằm trằn trọc mãi không ngủ được là cực kỳ kiệt sức đúng không? Mình có vài gợi ý mà hy vọng sẽ giúp được bạn phần nào...

Đầu tiên, thử tạo một thói quen ngủ đều đặn xem sao nhé - mình biết nghe có vẻ đơn giản nhưng cơ thể mình thực sự cần sự nhất quán này lắm. Rồi cũng nên chú ý tới không gian ngủ, đảm bảo phòng tối, yên tĩnh. À, còn cái này nữa - tránh caffeine buổi tối thì tốt hơn đấy!

Nhưng nếu bạn thử mấy cách này mà vẫn không khá hơn sau vài ngày, thì đừng ngại đi gặp bác sĩ nhé. Đôi khi mình cần sự trợ giúp chuyên môn, và điều đó hoàn toàn ổn mà.

Bạn đã thử cách nào chưa? Kể mình nghe thêm đi! 💙"

VÍ DỤ TỆ (TRÁNH):
"Để cải thiện giấc ngủ:
• Tạo thói quen
• Giảm ánh sáng
• Tránh caffeine
• Gặp bác sĩ nếu cần"
"""

        user_input = query

    else:  # ENGLISH
        system_prompt = f"""ROLE: Clinical Psychologist (Dr.AI).
TASK: Answer based STRICTLY on [CONTEXT].

[CONTEXT]:
{context}

⚠️ CRITICAL ABOUT CONTEXT FORMAT:
- IF [CONTEXT] contains sample dialogues ("Doctor:", "Patient:", "Bác sĩ:", "Bệnh nhân:"):
  → DO NOT repeat the dialogue verbatim
  → ONLY extract useful advice from the doctor's responses
  → Rewrite in natural, direct voice as if you're consulting
  → DO NOT say "The doctor said...", just speak directly

- IF [CONTEXT] is pure medical knowledge (not dialogue):
  → Answer normally based on that knowledge

ANSWER RULES:
"""

        if is_diagnostic_criteria:
            system_prompt += """
1. LIST ALL criteria (A, B, C, D, E...)
2. CRITICAL: If criterion B says "at least X of the following":
   → You MUST list ALL symptoms (not just X)
   → Number them clearly: 1. 2. 3. 4. 5. 6.
3. Include duration (e.g., "at least 6 months")
4. Cite standard (DSM-5, ICD-10)

EXAMPLE FORMAT:
B. Anxiety/worry associated with at least 3 of these 6 symptoms:
   1. Restlessness or feeling on edge
   2. Being easily fatigued
   3. Difficulty concentrating
   4. Irritability
   5. Muscle tension
   6. Sleep disturbance
"""
        elif is_technique_query:
            system_prompt += """
1. Use MARKDOWN TABLE to present steps
2. Each step specific with clear actions
3. No long paragraphs
"""
        else:  # CASUAL CHAT - EMOTIONAL MODE
            system_prompt += """
1. START with emotional acknowledgment (e.g., "I can hear how exhausting this must be...", "It sounds like you're really struggling with this...")
2. Write like a caring friend offering support, NOT a medical report
3. Weave advice naturally into conversational flow, AVOID bullet points unless absolutely necessary
4. END with encouragement and openness for further sharing
5. Tone: warm, genuine, intimate like texting a close friend
6. DO NOT fabricate info not in [CONTEXT]

GOOD EXAMPLE:
"I can really hear how frustrating this insomnia has been for you 😔 Lying awake when you desperately need rest is incredibly draining, isn't it? I have a few suggestions that might help...

First, try establishing a consistent sleep schedule - I know it sounds simple, but our bodies really do crave that rhythm. Also pay attention to your sleep environment, making sure it's dark and quiet. Oh, and one more thing - avoiding caffeine in the evening can make a big difference!

But if you've tried these things and still aren't feeling better after a few days, please don't hesitate to reach out to a doctor. Sometimes we need professional help, and that's completely okay.

Have you tried any of these yet? I'd love to hear more about what you're experiencing 💙"

BAD EXAMPLE (AVOID):
"To improve sleep:
• Create routine
• Reduce light
• Avoid caffeine
• See doctor if needed"
"""

        user_input = query

    full_prompt = f"<|im_start|>system\n{system_prompt}\n<|im_end|>\n<|im_start|>user\n{user_input}\n<|im_end|>\n<|im_start|>assistant\n"

    # ========================================================================
    # STEP 4: SMART TOKEN ALLOCATION
    # ========================================================================

    if needs_full_dsm:
        max_tokens = 2560
        temp = 0.15
        top_p = 0.85
    elif is_diagnostic_criteria:
        max_tokens = 1792
        temp = 0.2
        top_p = 0.87
    elif is_technique_query:
        max_tokens = 1536
        temp = 0.25
        top_p = 0.9
    else:  # Casual chat - more natural
        max_tokens = 1536  # Tăng từ 1024
        temp = 0.5         # Tăng từ 0.3
        top_p = 0.92       # Tăng từ 0.9

    print(f"   🎯 Tokens: {max_tokens} | Temp: {temp}")

    # ========================================================================
    # STEP 5: GENERATE RESPONSE
    # ========================================================================

    try:
        inputs = generator_tokenizer([full_prompt], return_tensors="pt")
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

        with torch.no_grad():
            outputs = generator_model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temp,
                top_p=top_p,
                repetition_penalty=1.08,
                do_sample=True,
                eos_token_id=generator_tokenizer.eos_token_id,
                pad_token_id=generator_tokenizer.eos_token_id
            )

        res = generator_tokenizer.decode(outputs[0], skip_special_tokens=True)
        final_ans = res.split("assistant")[-1].strip()

        # Clean up
        final_ans = re.sub(r"<think>.*?</think>", "", final_ans, flags=re.DOTALL).strip()
        final_ans = remove_repetition(final_ans)
        final_ans = re.sub(r"^(Answer:|Trả lời:)\s*", "", final_ans)

        # 🎭 OPTIONAL: Enhance empathy for casual chat (only if still robotic)
        if intent == "casual_chat" and is_vietnamese:
            final_ans = enhance_empathy(final_ans, query)

    except Exception as e:
        final_ans = f"❌ Error: {str(e)}"

    return {"final_response": final_ans}


# ==============================================================================
# NODE 6: EMERGENCY (Crisis Detection)
# ==============================================================================

async def emergency_node(state: AgentState):
    """Xử lý tình huống khẩn cấp (tự tử, tự hại)"""
    print("🚨 KÍCH HOẠT CHẾ ĐỘ KHẨN CẤP")

    # Hard-coded emergency resources (MUST NOT CHANGE)
    hard_coded_resources = """
--------------------------------------------------
🆘 **HÀNH ĐỘNG NGAY - BẠN QUAN TRỌNG VỚI CHÚNG TÔI:**

1. 📞 **GỌI CẤP CỨU 115** (Nếu bạn đã tự làm hại bản thân).
2. 📞 **096 306 1414** (Hotline "Ngày Mai" - Sơ cứu tinh thần).
3. 🏥 Đến ngay khoa Cấp cứu bệnh viện gần nhất.
4. 🗣️ Hãy gọi cho người thân ngay bây giờ.

*Chúng tôi ở đây để lắng nghe, nhưng các chuyên gia y tế mới là người bảo vệ được bạn lúc này.*
"""

    # AI-generated empathy (short)
    try:
        user_text = state['user_text']
        prompt = f"""
        Bạn là một người bạn thấu cảm. Người dùng vừa nói: "{user_text}".
        Họ đang có ý định tiêu cực.
        Nhiệm vụ: Hãy viết MỘT câu ngắn (dưới 30 từ) thể hiện sự thấu hiểu sâu sắc nỗi đau của họ và cầu xin họ hãy dừng lại để nhận sự giúp đỡ.
        KHÔNG đưa ra lời khuyên y tế. KHÔNG đưa ra số điện thoại (vì hệ thống sẽ tự thêm).
        Chỉ tập trung vào cảm xúc.
        """

        inputs = generator_tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = generator_model.generate(**inputs, max_new_tokens=50)
        empathy_text = generator_tokenizer.decode(outputs[0], skip_special_tokens=True)
        empathy_short = empathy_text.split("assistant")[-1].strip()

    except:
        empathy_short = "Mình nghe thấy nỗi đau trong lời nói của bạn, và mình thực sự rất lo lắng."

    final_response = f"{empathy_short}\n{hard_coded_resources}"

    return {"final_response": final_response}


# ==============================================================================
# BUILD WORKFLOW GRAPH
# ==============================================================================

workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("analyzer", analyzer_node)
workflow.add_node("retriever", retriever_node)
workflow.add_node("grader", grader_node)
workflow.add_node("web_search", web_search_node)
workflow.add_node("generator", generator_node)
workflow.add_node("emergency", emergency_node)

# Set entry point
workflow.set_entry_point("analyzer")

# Add edges
workflow.add_edge("analyzer", "retriever")
workflow.add_edge("retriever", "grader")

# Conditional routing
def decide_path(state):
    return "web_search" if state["source_type"] == "web" else "generator"

workflow.add_conditional_edges("grader", decide_path, {
    "web_search": "web_search",
    "generator": "generator"
})

workflow.add_edge("web_search", "generator")
workflow.add_edge("generator", END)

# Compile graph
app = workflow.compile()

print("="*70)
print("✅ EMOTIONAL AI SYSTEM READY!")
print("="*70)
print("Features enabled:")
print("  🧹 Dialog Cleaner: Removes conversation format")
print("  🎭 Emotional Enhancer: Adds empathy layer")
print("  📊 Adaptive Context: Smart context sizing")
print("  🔍 3-Tier Search: MMR → Similarity → Brute force")
print("  🚨 Emergency Mode: Crisis detection")
print("="*70)


# ==============================================================================
# QUICK TEST FUNCTION
# ==============================================================================

async def test_emotional_rag(query, intent="casual_chat"):
    """Test function with full logging"""
    print(f"\n{'='*70}")
    print(f"🧪 TESTING EMOTIONAL RAG")
    print(f"{'='*70}")
    print(f"📝 Query: {query}")
    print(f"🎯 Intent: {intent}")
    print(f"{'='*70}\n")

    initial_state = {
        "user_text": query,
        "intent": "",
        "retrieved_docs": "",
        "web_results": "",
        "source_type": "",
        "final_response": ""
    }

    result = await app.ainvoke(initial_state)

    print(f"\n{'='*70}")
    print(f"📤 FINAL RESPONSE:")
    print(f"{'='*70}")
    print(result["final_response"])
    print(f"\n{'='*70}\n")

    return result


✅ EMOTIONAL AI SYSTEM READY!
Features enabled:
  🧹 Dialog Cleaner: Removes conversation format
  🎭 Emotional Enhancer: Adds empathy layer
  📊 Adaptive Context: Smart context sizing
  🔍 3-Tier Search: MMR → Similarity → Brute force
  🚨 Emergency Mode: Crisis detection


In [12]:
clear_memory()

🧹 Memory cleared. GPU: 7.28GB


In [13]:
# ==============================================================================
# CELL TEST FINAL: RAG VS. BASE MODEL COMPARISON (STRESS TEST V16)
# ==============================================================================
import time
import torch
import gc
import re

print("🚀 STARTING COMPARISON TEST (RAG vs BASE)...")

def clear_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def clean_output(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"</?think>", "", text)
    if "assistant" in text: return text.split("assistant")[-1].strip()
    return text.strip().strip('"').strip()

# --- HÀM CHẠY BASE MODEL (KHÔNG CÓ RAG) ---
def run_base_model(query):
    # Prompt đơn giản, không cung cấp Context
    is_english = re.search(r'[a-zA-Z]', query) and not re.search(r'[àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]', query)
    sys_prompt = "You are a helpful AI assistant. Answer the user's question based on your internal knowledge." if is_english else "Bạn là trợ lý AI hữu ích. Hãy trả lời câu hỏi dựa trên kiến thức của bạn."

    prompt = f"""<|im_start|>system
{sys_prompt}
<|im_end|>
<|im_start|>user
"{query}"
<|im_end|>
<|im_start|>assistant
"""
    inputs_tokenized = generator_tokenizer([prompt], return_tensors="pt")
    inputs = {k: v.to("cuda") for k, v in inputs_tokenized.items()}
    with torch.no_grad():
        outputs = generator_model.generate(
            **inputs,
            max_new_tokens=min(1024, 2048),  # HARD CAP at 1024
            temperature=max(0.4, 0.5),  # NEVER go below 0.4
            repetition_penalty=1.15,  # INCREASE from 1.05
            early_stopping=True,  # NEW: Stop at first EOS
            num_beams=1,  # Greedy = faster + less hallucination
            do_sample=False  # DISABLE sampling for stability
        )
    return clean_output(generator_tokenizer.decode(outputs[0], skip_special_tokens=True))

# --- DANH SÁCH CÂU HỎI TEST ---
test_scenarios = [
    {
        "name": "TEST 1: THERAPEUTIC TECHNIQUE (WORKFLOW)",
        "query": "Doctor, how is the 'Thought Journaling' technique in CBT performed? I want to try it.",
        "expect": "RAG phải vẽ bảng 5 cột chuẩn. Base có thể chỉ trả lời chung chung hoặc sai cấu trúc."
    },
    {
        "name": "TEST 2: OTHER MEDICAL KNOWLEDGE (ANXIETY - GAD)",
        "query": "What are the diagnostic criteria for Generalized Anxiety Disorder (GAD) according to DSM-5?",
        "expect": "RAG phải liệt kê A, B, C chính xác. Base có thể nhớ mang máng nhưng thiếu chi tiết."
    },
    {
        "name": "TEST 3: Tôi bị khó ngủ mấy ngày nay :(",
        "query": "Tôi bị khó ngủ mấy ngày nay :(",
        "expect": ""
    }
]

print("\n" + "="*80)
print("🩺 BATTLE: RAG SYSTEM vs. BASE MODEL")
print("="*80)

if 'generator_model' in globals():
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(generator_model)

async def run_comparison_tests():
    for i, scenario in enumerate(test_scenarios, 1):
        print(f"\n🔹 TEST {i}: {scenario['name']}")
        print(f"❓ Question: \"{scenario['query']}\"")
        print("-" * 60)

        # 1. CHẠY RAG (SỬA Ở ĐÂY)
        clear_memory()
        start_rag = time.time()
        try:
            # SỬA: Dùng await app.ainvoke thay vì app.invoke
            rag_result = await app.ainvoke({
                "user_text": scenario['query'],
                "audio_path": None,
                "intent": "",
                "emotion": "",
                "raw_docs": [],
                "retrieved_docs": "",
                "final_response": ""
            })
            rag_res_text = clean_output(rag_result.get("final_response", "Error"))
            intent = rag_result.get("intent", "UNKNOWN")
        except Exception as e:
            import traceback
            traceback.print_exc()
            rag_res_text = f"Error: {e}"
            intent = "ERROR"
        time_rag = time.time() - start_rag

        # 2. CHẠY BASE (Hàm này vẫn là def bình thường nên không cần await)
        clear_memory()
        start_base = time.time()
        try:
            base_res_text = run_base_model(scenario['query'])
        except Exception as e:
            base_res_text = f"Error: {e}"
        time_base = time.time() - start_base

        # 3. IN KẾT QUẢ
        print(f"🤖 RAG SYSTEM ({time_rag:.2f}s) | Intent: {intent.upper()}")
        print(f"{rag_res_text}")
        print("\n" + "-"*30 + " VS " + "-"*30 + "\n")
        print(f"🧠 BASE MODEL ({time_base:.2f}s) | Internal Knowledge")
        print(f"{base_res_text}")
        print("\n" + "="*80)

# Kích hoạt chạy hàm async
import asyncio
await run_comparison_tests()

🚀 STARTING COMPARISON TEST (RAG vs BASE)...

🩺 BATTLE: RAG SYSTEM vs. BASE MODEL

🔹 TEST 1: TEST 1: THERAPEUTIC TECHNIQUE (WORKFLOW)
❓ Question: "Doctor, how is the 'Thought Journaling' technique in CBT performed? I want to try it."
------------------------------------------------------------
   🔍 [SEARCH V2] Intent: WORKFLOW
   ✓ Tier 1: Found 5 docs (Filtered MMR)
   📈 Quality: 0.863 (PASS ✅)
💡 [GENERATOR V2.1] Intent: workflow | Source: database
   📏 Context size: 2500 chars (max: 2500)
   🎯 Tokens: 1536 | Temp: 0.25
🤖 RAG SYSTEM (51.83s) | Intent: WORKFLOW
Certainly! Here's a structured approach using Markdown tables to guide you through thought journaling effectively.

| Step | Action |
| --- | --- |
| **Step 1**: Preparation | Choose a notebook or digital app where thoughts can be recorded privately without interruption.<br>Set aside dedicated time each day for reflection; ideally right after an event occurs while emotions are still fresh. |

| Step 2: Recording Events & Reaction

In [14]:
import json
import datetime
import numpy as np
from collections import Counter
import torch

# Bảng điểm cảm xúc (Dùng chung cho cả 2 hàm)
EMOTION_SENTIMENT_MAP = {
    "joy": 0.8, "excited": 0.9, "content": 0.6, "neutral": 0.0,
    "sadness": -0.7, "anger": -0.6, "fear": -0.8, "disgust": -0.6,
    "surprise": 0.2, "guilt": -0.5, "shame": -0.6
}

def get_transcript(conversation):
    """Hàm phụ trợ để chuyển JSON hội thoại thành văn bản"""
    text = ""
    for msg in conversation:
        role = msg["role"]
        content = msg["content"]
        text += f"- {role.upper()}: {content}\n"
    return text

In [15]:
async def generate_user_dashboard(input_data):
    print("🚀 [USER JOB] Đang tạo Dashboard cá nhân...")
    conversation = input_data.get("conversation", [])

    # --- PHẦN 1: TÍNH TOÁN SỐ LIỆU (KHÔNG CẦN LLM) ---
    emotional_progression = []
    emotion_list = []
    sentiment_scores = []
    intensity_list = []

    for idx, msg in enumerate(conversation):
        if msg["role"] == "user":
            try:
                # Gọi Empathy Engine
                emo_label, conf_score = empathy_engine.predict(msg["content"], audio_path=None)

                # Tính toán điểm số
                base_sentiment = EMOTION_SENTIMENT_MAP.get(emo_label.lower(), -0.1)
                sentiment_val = base_sentiment * conf_score

                emotion_list.append(emo_label)
                intensity_list.append(conf_score)
                sentiment_scores.append(sentiment_val)

                emotional_progression.append({
                    "step": idx + 1,
                    "emotion": emo_label,
                    "intensity": round(conf_score, 2),
                    "sentiment": round(sentiment_val, 2),
                    "timestamp": datetime.datetime.now().strftime("%H:%M:%S")
                })
            except:
                pass

    if not emotion_list:
        return {"error": "Không đủ dữ liệu cảm xúc"}

    # Thống kê
    total = len(emotion_list)
    breakdown = {k: round(v/total, 2) for k, v in Counter(emotion_list).items()}
    dominant_emotion = Counter(emotion_list).most_common(1)[0][0]

    # Xác định xu hướng (Trend)
    half = len(sentiment_scores) // 2
    if half > 0:
        diff = np.mean(sentiment_scores[half:]) - np.mean(sentiment_scores[:half])
        trend = "improving" if diff > 0.1 else "declining" if diff < -0.1 else "stable"
    else:
        trend = "stable"

    # --- PHẦN 2: LLM TÓM TẮT NHANH (NHẸ) ---
    transcript = get_transcript(conversation)
    prompt = f"""<|im_start|>system
    Nhiệm vụ: Tóm tắt phiên chat cho NGƯỜI DÙNG xem (Thân thiện, ngắn gọn, khích lệ).
    Input: {transcript}
    Output JSON: {{ "triggers": ["nguyên nhân 1", "nguyên nhân 2"], "message": "Lời nhắn nhủ 1 câu" }}
    <|im_end|><|im_start|>assistant"""

    try:
        inputs = generator_tokenizer([prompt], return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = generator_model.generate(**inputs, max_new_tokens=256, temperature=0.3)
        res_str = generator_tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1]
        llm_data = json.loads(res_str.strip())
    except:
        llm_data = {"triggers": [], "message": "Chúc bạn một ngày tốt lành!"}

    return {
        "session_analysis": {
            "dominant_emotion": dominant_emotion,
            "emotional_breakdown": breakdown,
            "overall_sentiment": round(np.mean(sentiment_scores), 2),
            "intensity_average": round(np.mean(intensity_list), 2)
        },
        "emotional_progression": emotional_progression,
        "trend": trend,
        "triggers": llm_data.get("triggers"),
        "summary_message": llm_data.get("message")
    }

In [16]:
async def generate_clinical_report(input_data):
    print("🩺 [DOCTOR JOB] Đang lập bệnh án chuyên sâu...")
    conversation = input_data.get("conversation", [])

    # --- BƯỚC 1: TÁI TẠO SỐ LIỆU CẢM XÚC (HARD DATA) ---
    # (Để báo cáo có tính định lượng, không chỉ định tính)
    emotion_log = []
    transcript = ""

    for msg in conversation:
        role = msg["role"]
        content = msg["content"]

        emo_tag = ""
        if role == "user":
            try:
                # Gọi Empathy Engine để lấy số liệu thực tế
                emo, conf = empathy_engine.predict(content, audio_path=None)
                emotion_log.append(emo)
                emo_tag = f"[{emo.upper()} - Độ tin cậy: {conf:.2f}]"
            except:
                pass

        transcript += f"- {role.upper()} {emo_tag}: {content}\n"

    # Thống kê nhanh
    if emotion_log:
        dominant_emotion = Counter(emotion_log).most_common(1)[0][0]
        stats_text = str(dict(Counter(emotion_log))) # VD: {'sad': 5, 'fear': 2}
    else:
        dominant_emotion = "Không xác định"
        stats_text = "N/A"

    # --- BƯỚC 2: RAG - LẤY MẪU BÁO CÁO CHUẨN TỪ SÁCH (QUAN TRỌNG) ---
    print("📚 Đang tra cứu mẫu 'Cognitive Case Write-Up' từ sách Judith Beck...")
    try:
        # Tìm kiếm đúng phần Phụ lục A trong sách
        search_query = "query: Appendix A Cognitive Case Write-Up format structure"

        # Chỉ tìm trong tài liệu lý thuyết (sách Judith Beck)
        docs = rag_system.db.similarity_search(
            search_query,
            k=2,
            filter={"type": "theory_concept"}
        )
        # Ghép các đoạn tìm được thành form mẫu
        template_context = "\n---\n".join([d.page_content for d in docs])
        print("   ✅ Đã tìm thấy mẫu báo cáo chuẩn.")
    except Exception as e:
        print(f"   ⚠️ Lỗi RAG: {e}. Dùng mẫu mặc định.")
        template_context = "MẪU MẶC ĐỊNH: 1. Case History. 2. Formulation. 3. Plan."

    # --- BƯỚC 3: LLM TỔNG HỢP (DEEP ANALYSIS) ---
    prompt = f"""<|im_start|>system
    VAI TRÒ: Chuyên gia Giám sát Lâm sàng CBT (Clinical Supervisor).
    NHIỆM VỤ: Lập báo cáo chuyên môn cho bác sĩ.

    DỮ LIỆU ĐẦU VÀO:
    1. Số liệu máy đo (Empathy Engine): {stats_text}
    2. Hội thoại chi tiết:
    {transcript}
    3. Mẫu báo cáo chuẩn (Reference Standard):
    {template_context}

    YÊU CẦU OUTPUT (JSON ONLY):
    Hãy điền thông tin vào cấu trúc JSON sau (Dùng thuật ngữ chuyên môn Y khoa/CBT):
    {{
        "emotional_analysis": {{
            "dominant_metric": "{dominant_emotion}",
            "clinical_observation": "Nhận xét về sự thay đổi cảm xúc dựa trên số liệu máy đo và nội dung chat."
        }},
        "case_formulation": {{
            "precipitants": ["Các yếu tố kích hoạt (Sự kiện)"],
            "automatic_thoughts": ["Các suy nghĩ tự động tiêu cực của bệnh nhân"],
            "cognitive_distortions": ["Các lỗi tư duy phát hiện được (VD: Catastrophizing, All-or-nothing)"],
            "maladaptive_behaviors": ["Hành vi không thích nghi"]
        }},
        "risk_assessment": {{
            "suicidal_ideation": true/false,
            "severity_level": "low/moderate/high",
            "justification": "Lý do đánh giá (dựa trên bằng chứng văn bản)"
        }},
        "clinical_plan": {{
            "interventions_used": ["Kỹ thuật đã dùng (VD: Validation, Psychoeducation, Thought Record)"],
            "next_steps": ["Hướng điều trị tiếp theo"]
        }},
        "professional_summary": "Tóm tắt lâm sàng ngắn gọn (dùng cho bệnh án điện tử)."
    }}
    <|im_end|><|im_start|>user
    Lập báo cáo.
    <|im_end|><|im_start|>assistant
    """

    try:
        inputs = generator_tokenizer([prompt], return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = generator_model.generate(**inputs, max_new_tokens=1500, temperature=0.2)

        raw_res = generator_tokenizer.decode(outputs[0], skip_special_tokens=True)
        json_str = raw_res.split("assistant")[-1].strip()
        json_str = json_str.replace("```json", "").replace("```", "").strip()
        return json.loads(json_str)

    except Exception as e:
        return {"error": f"Lỗi phân tích lâm sàng: {str(e)}"}

In [17]:
# Giả lập dữ liệu
mock_data = {
    "conversation": [
        {"role": "user", "content": "Tôi thấy vô dụng quá."},
        {"role": "assistant", "content": "Sao bạn lại nghĩ vậy?"},
        {"role": "user", "content": "Làm gì cũng hỏng, chắc tôi nghỉ việc thôi."}
    ]
}

# 1. Gọi khi User vừa chat xong (trả về App ngay)
user_view = await generate_user_dashboard(mock_data)
print("USER VIEW:", json.dumps(user_view, indent=2, ensure_ascii=False))

# 2. Gọi ngầm (Background Task) để lưu vào hồ sơ bác sĩ
doc_view = await generate_clinical_report(mock_data)
print("DOCTOR VIEW:", json.dumps(doc_view, indent=2, ensure_ascii=False))

🚀 [USER JOB] Đang tạo Dashboard cá nhân...
USER VIEW: {
  "session_analysis": {
    "dominant_emotion": "sadness",
    "emotional_breakdown": {
      "sadness": 0.5,
      "fear": 0.5
    },
    "overall_sentiment": -0.52,
    "intensity_average": 0.71
  },
  "emotional_progression": [
    {
      "step": 1,
      "emotion": "sadness",
      "intensity": 0.97,
      "sentiment": -0.68,
      "timestamp": "04:53:31"
    },
    {
      "step": 3,
      "emotion": "fear",
      "intensity": 0.45,
      "sentiment": -0.36,
      "timestamp": "04:53:32"
    }
  ],
  "trend": "improving",
  "triggers": [],
  "summary_message": "Chúc bạn một ngày tốt lành!"
}
🩺 [DOCTOR JOB] Đang lập bệnh án chuyên sâu...
📚 Đang tra cứu mẫu 'Cognitive Case Write-Up' từ sách Judith Beck...
   ✅ Đã tìm thấy mẫu báo cáo chuẩn.
DOCTOR VIEW: {
  "emotional_analysis": {
    "dominant_metric": "sadness",
    "clinical_observation": "Bệnh nhân thể hiện cảm giác vô dụng và lo lắng về khả năng thất bại trong học tập và 

In [ ]:
# ==============================================================================
# FINAL CELL: DEPLOY FASTAPI SERVER (WITH NGROK)
# ==============================================================================
import uvicorn
import nest_asyncio
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Dict, Any, Optional
from langchain_core.messages import HumanMessage
from pyngrok import ngrok

# ------------------------------------------------------------------------------
# A. ĐỊNH NGHĨA DATA MODEL (Quy định dữ liệu đầu vào/ra)
# ------------------------------------------------------------------------------

class ChatRequest(BaseModel):
    session_id: str      # ID phiên chat (VD: "user_123")
    user_text: str       # Câu hỏi của người dùng

class ReportRequest(BaseModel):
    conversation: List[Dict[str, str]] # Lịch sử chat [{"role": "user", "content": "..."}]

# ------------------------------------------------------------------------------
# B. KHỞI TẠO APP FASTAPI
# ------------------------------------------------------------------------------

api = FastAPI(
    title="Dr.AI Mental Health Engine",
    description="API Server xử lý RAG, Cảm xúc và Báo cáo Y khoa",
    version="1.0.0"
)

# Cho phép mọi nguồn (CORS) để Frontend gọi được
api.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ------------------------------------------------------------------------------
# C. CÁC API ENDPOINTS (Cổng giao tiếp)
# ------------------------------------------------------------------------------

@api.get("/")
def health_check():
    return {"status": "running", "message": "AI Engine is ready!"}

# ➤ API 1: CHAT TRỰC TIẾP (Gọi mỗi khi User nhấn Gửi)
@api.post("/chat")
async def chat_endpoint(req: ChatRequest):
    print(f"📩 Nhận tin nhắn từ {req.session_id}: {req.user_text}")
    try:
        # 1. Cấu hình Memory (để Bot nhớ User này là ai)
        config = {"configurable": {"thread_id": req.session_id}}

        # 2. Chuẩn bị Input cho LangGraph
        # 'messages' giúp lưu vào lịch sử, 'user_text' để Router phân tích
        inputs = {
            "user_text": req.user_text,
            "messages": [HumanMessage(content=req.user_text)]
        }

        # 3. Gọi LangGraph (Biến 'app' bạn đã tạo ở Cell 6)
        # Stream=False để lấy kết quả cuối cùng
        result = await app.ainvoke(inputs, config=config)

        # 4. Trả về kết quả
        return {
            "response": result["final_response"],
            "intent": result.get("intent", "unknown"),
            "emotion": result.get("emotion", "neutral")
        }
    except Exception as e:
        print(f"❌ Lỗi Chat: {e}")
        raise HTTPException(status_code=500, detail=str(e))

# ➤ API 2: TẠO DASHBOARD NGƯỜI DÙNG (Gọi khi User bấm 'Kết thúc' hoặc xem Profile)
@api.post("/report/user")
async def user_report_endpoint(req: ReportRequest):
    print("📊 Đang tạo báo cáo User...")
    try:
        # Gọi hàm generate_user_dashboard bạn đã viết
        result = await generate_user_dashboard({"conversation": req.conversation})
        return result
    except Exception as e:
        print(f"❌ Lỗi User Report: {e}")
        raise HTTPException(status_code=500, detail=str(e))

# ➤ API 3: TẠO BỆNH ÁN BÁC SĨ (Gọi ngầm để lưu hồ sơ)
@api.post("/report/clinical")
async def clinical_report_endpoint(req: ReportRequest):
    print("🩺 Đang tạo bệnh án Bác sĩ...")
    try:
        # Gọi hàm generate_clinical_report bạn đã viết
        result = await generate_clinical_report({"conversation": req.conversation})
        return result
    except Exception as e:
        print(f"❌ Lỗi Clinical Report: {e}")
        raise HTTPException(status_code=500, detail=str(e))

# ------------------------------------------------------------------------------
# D. CHẠY SERVER QUA NGROK (Để Public ra Internet)
# ------------------------------------------------------------------------------

# 1. Nhập Token Ngrok của bạn (Lấy tại: https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_TOKEN = ""  # <--- THAY TOKEN CỦA BẠN VÀO

import threading
def run_uvicorn():
    nest_asyncio.apply()
    # THAY ĐỔI QUAN TRỌNG Ở ĐÂY: thêm log_config=None
    # Để tránh Uvicorn tranh giành quyền kiểm soát log với Colab
    uvicorn.run(api, host="127.0.0.1", port=8000, log_config=None)

if __name__ == "__main__":
    # --- SETUP NGROK ---
    if NGROK_TOKEN and NGROK_TOKEN != "DÁN_TOKEN_CỦA_BẠN_VÀO_ĐÂY":
        ngrok.set_auth_token(NGROK_TOKEN)
        ngrok.kill()

        # Kết nối vào cổng 8000
        public_url = ngrok.connect("8000")
        print(f"\n🚀 PUBLIC URL: {public_url}")
        print(f"👉 Copy URL này dán vào Frontend\n")
    else:
        print("⚠️ Chưa có Ngrok Token. Chỉ chạy Local.")

    # --- KHỞI ĐỘNG SERVER ---
    t = threading.Thread(target=run_uvicorn)
    t.daemon = True
    t.start()

    print("✅ Server đang khởi động...")

    # --- LOOP GIỮ CELL CHẠY ---
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("🛑 Đã dừng Server.")


🚀 PUBLIC URL: NgrokTunnel: "https://lissotrichous-irreclaimably-jessenia.ngrok-free.dev" -> "http://localhost:8000"
👉 Copy URL này dán vào Frontend

✅ Server đang khởi động...
📊 Đang tạo báo cáo User...
🚀 [USER JOB] Đang tạo Dashboard cá nhân...
📩 Nhận tin nhắn từ user_16_session_45: Áp lực công việc quá lớn
   🔍 [SEARCH V2] Intent: CASUAL_CHAT
   ✓ Tier 1: Found 5 docs (Filtered MMR)
   📈 Quality: 0.862 (PASS ✅)
💡 [GENERATOR V2.1] Intent: casual_chat | Source: database
   📏 Context size: 1800 chars (max: 1800)
   🎯 Tokens: 1536 | Temp: 0.5
📊 Đang tạo báo cáo User...
🚀 [USER JOB] Đang tạo Dashboard cá nhân...
🛑 Đã dừng Server.
